# BSM Paper Replication — Notebook 08: VGG16 (paper-faithful, real run)

60 epochs, full-batch grad-accum (micro 32, one optim step/epoch), Adam 1e-3, wd 0,
best-validation checkpoint, evaluated on the full 15 % test split.

**Param count note:** `vgg16(head='paper')` = 13 conv layers (2,2,3,3,3 blocks; 64,128,256,512,512) +
FC(4096) -> FC(4096) -> FC(6), dropout 0.5, no batch norm = **138,357,544** params
(DEVIATIONS row 12: the paper's Table 4 is incomplete, the prose describes standard VGG16).


In [ ]:
# setup
import os, sys
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"): drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
if PROJECT not in sys.path: sys.path.insert(0, PROJECT)
import torch, random, numpy as np
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("gpu:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")


## Real run: VGG16, 60 epochs, full-batch grad-accum (micro 32, one optim step/epoch), Adam 1e-3, best-val checkpoint

In [ ]:
# ===== REAL TRAINING: VGG16 (60 epochs, paper-faithful) =====
import os, sys, time, json, pathlib
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
VER = pathlib.Path(PROJECT) / "versions"
CKPT = pathlib.Path(PROJECT) / "checkpoints" / "vgg16"
RES  = pathlib.Path(PROJECT) / "results"
CKPT.mkdir(parents=True, exist_ok=True)
RES.mkdir(parents=True, exist_ok=True)
if PROJECT not in sys.path: sys.path.insert(0, PROJECT)

from models import vgg16
import common

def load_xy(prefix):
    X = np.load(VER / f"X_{prefix}.npy")
    y = np.load(VER / f"y_{prefix}.npy")
    Xt = torch.from_numpy(X.transpose(0, 3, 1, 2)).contiguous()  # NHWC -> NCHW
    yt = torch.from_numpy(y.astype(np.int64))
    return Xt, yt

Xtr, ytr = load_xy("train")
Xva, yva = load_xy("val")
print("train:", Xtr.shape, "val:", Xva.shape, "| dtype", Xtr.dtype, "range", float(Xtr.min()), float(Xtr.max()))

MICRO = 32
train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=MICRO, shuffle=False)
val_loader   = DataLoader(TensorDataset(Xva, yva), batch_size=MICRO, shuffle=False)

torch.manual_seed(42)
model = vgg16().cuda()
n_params = sum(p.numel() for p in model.parameters())
print("vgg16 params:", f"{n_params:,}")

EPOCHS = 60
t0 = time.time()
hist = common.train_model(
    model, train_loader, val_loader, torch.device("cuda"),
    epochs=EPOCHS, full_batch=True,
    ckpt_dir=CKPT, history_path=RES / "history_vgg16.json",
    tag="vgg16", lr=1e-3, resume=True,
)
print(f"TRAINING DONE in {(time.time()-t0)/60:.1f} min")
print("best epoch:", hist.get("best_epoch"), "best val:", hist.get("best_val_acc"))

tr_acc = hist.get("train_acc") or []
if len(tr_acc) >= 30 and tr_acc[29] < 0.30:
    print("WARNING: train acc at epoch 30 =", tr_acc[29], "< 0.30 (spec stop-check)")


In [ ]:
# ===== EVAL VGG16 (best-val ckpt) on TEST SPLIT =====
import os, sys, json, pathlib
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
VER = pathlib.Path(PROJECT) / "versions"
CKPT = pathlib.Path(PROJECT) / "checkpoints" / "vgg16"
RES  = pathlib.Path(PROJECT) / "results"
if PROJECT not in sys.path: sys.path.insert(0, PROJECT)
from models import vgg16
import common

CLASSES = ["Ilmenite","Rutile","Zircon","Garnet","Monazite","Sillimanite"]

def load_xy(prefix):
    X = np.load(VER / f"X_{prefix}.npy")
    y = np.load(VER / f"y_{prefix}.npy")
    return torch.from_numpy(X.transpose(0,3,1,2)).contiguous(), torch.from_numpy(y.astype(np.int64))

Xte, yte = load_xy("test")
print("test:", Xte.shape)
test_loader = DataLoader(TensorDataset(Xte, yte), batch_size=32, shuffle=False)

device = torch.device("cuda")
model = vgg16().to(device)
ck = torch.load(CKPT / "vgg16_best.pt", map_location=device, weights_only=True)
model.load_state_dict(ck)
_hist = json.load(open(RES / "history_vgg16.json"))
_ck = {"epoch": _hist.get("best_epoch", -1)}
model.eval()

all_preds, all_true = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        logits = model(xb.to(device))
        all_preds.append(logits.argmax(1).cpu().numpy())
        all_true.append(yb.numpy())
y_pred = np.concatenate(all_preds); y_true = np.concatenate(all_true)

acc = accuracy_score(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
cm = confusion_matrix(y_true, y_pred, labels=range(6))
spec_list = []
for i in range(6):
    tn = cm.sum() - cm[i,:].sum() - cm[:,i].sum() + cm[i,i]
    fp = cm[:,i].sum() - cm[i,i]
    spec_list.append(tn/(tn+fp) if (tn+fp)>0 else 0.0)
spec = float(np.mean(spec_list))

print(f"\n=== VGG16 TEST SPLIT (best epoch {_ck['epoch']}) ===")
print(f"accuracy {acc:.4f} | macro precision {prec:.4f} | macro recall {rec:.4f} | macro F1 {f1:.4f} | macro specificity {spec:.4f}")
print("\nconfusion matrix (rows=true):")
print(cm)
print("\nper-class:")
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=4, zero_division=0))

results = {
  "model": "vgg16", "notebook": "08_vgg16",
  "best_epoch": int(_ck.get("epoch", -1)),
  "test_split": {"accuracy": float(acc), "macro_precision": float(prec), "macro_recall": float(rec),
                 "macro_f1": float(f1), "macro_specificity": spec,
                 "confusion_matrix": cm.tolist(), "n_test": int(len(y_true))},
  "paper_reported_table9": {"accuracy": 0.83, "precision": 0.89, "recall": 0.83, "note": "PAPER numbers (Table 9), for comparison only"},
}
with open(RES / "metrics_vgg16.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nsaved", RES / "metrics_vgg16.json")
print("EVAL DONE")


In [ ]:
# persist notebook copy
import glob, os, shutil
PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
os.makedirs(os.path.join(PROJECT,"notebooks"), exist_ok=True)
cands = glob.glob("/content/drive/MyDrive/Colab Notebooks/*.ipynb")
if cands:
    newest = max(cands, key=os.path.getmtime)
    shutil.copy(newest, os.path.join(PROJECT,"notebooks","08_vgg16.ipynb"))
    print("copied ->", os.path.join(PROJECT,"notebooks","08_vgg16.ipynb"))
